In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import StringIO

plt.style.use("dark_background")
default_colors = plt.rcParamsDefault['axes.prop_cycle']
plt.rcParams['axes.prop_cycle'] = default_colors

In [ ]:

# CSV URL
url = "https://ourworldindata.org/grapher/monthly-spending-data-center-us.csv?v=1&csvType=full&useColumnShortNames=false"

# Add browser-like headers
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Download CSV manually
response = requests.get(url, headers=headers)

# Raise error if request failed
response.raise_for_status()

# Load into pandas
# We use StringIO to read the CSV content from the response text - this allows us to treat the string as a file-like object for pandas to read.
df = pd.read_csv(StringIO(response.text))

# Rename columns
df = df.rename(columns={
    'Day': 'Date',
    'Monthly spending on data center construction in the United States': 'Spending_USD'
})

# Convert dates
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df = df.sort_values('Date')

# Convert units
df['Spending_Billions'] = df['Spending_USD'] / 1_000_000_000


In [ ]:
df

In [ ]:
##### Monthly trend
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['Spending_Billions'], linewidth=2)

plt.title('Monthly US Data Center Construction Spending')
plt.xlabel('Date')
plt.ylabel('Spending (Billions USD)')
plt.grid(alpha = 0.1)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Annual totals
annual = df.groupby('Year')['Spending_Billions'].sum().round(2).reset_index()
annual

In [ ]:
annual_adj = annual.copy()
annual_adj.loc[annual_adj['Year']==2026,'Spending_Billions'] = (annual_adj.loc[annual_adj['Year']==2026,'Spending_Billions'] * 12).round(2)

In [ ]:
annual_adj

In [ ]:
plt.figure(figsize=(10, 5))

target_year = 2026
colours = ['#1f77b4' if year != target_year else "#ffa30e" for year in annual_adj['Year']]
plt.bar(annual_adj['Year'], annual_adj['Spending_Billions'], color=colours)

plt.title('Annual US Data Center Construction Spending (Adjusted)') 
plt.xlabel('Year')
plt.ylabel('Spending (Billions USD)')
plt.grid(alpha = 0.1, axis='y')
plt.xticks(annual_adj['Year'])
plt.tight_layout()

# --- Legend handles ---
normal_patch = plt.Rectangle((0,0), 1, 1, color='#1f77b4')
highlight_patch = plt.Rectangle((0,0), 1, 1, color='#ffa30e')

plt.legend(
    [normal_patch, highlight_patch],
    ['Adjusted Spending', f'Adjusted {target_year} Spending (Annualized from Jan 2026)']
)

plt.show()